# Postgres → Delta ETL Pipeline (Incremental)

Read videos từ Lakebase Postgres (do user insert qua Databricks App) → MERGE vào Delta table.

**Incremental Flow:**
1. Lấy watermark (`MAX(updated_at)`) từ Delta table
2. SELECT từ `lakebase_pg.public.videos` (foreign catalog): chỉ rows có `COALESCE(updated_at, created_at) > watermark`
3. MERGE vào `prod.default.youtube_videos_delta` (UPSERT on video_id)
4. Verify row count

**Kiến trúc:**
- Databricks App → user search YouTube → INSERT vào Postgres
- Job notebook (this) → SELECT Postgres via foreign catalog (incremental) → MERGE vào Delta (analytics)

> **Note:** Đọc qua foreign catalog `lakebase_pg` thay vì JDBC để tránh timezone mismatch giữa Spark (UTC) và Postgres.

In [0]:
# --- Config ---
PG_TABLE = "lakebase_pg.public.videos"  # Lakebase Postgres foreign catalog
DELTA_TABLE = "prod.default.youtube_videos_delta"

print(f"Postgres source: {PG_TABLE}")
print(f"Delta target: {DELTA_TABLE}")

In [0]:
# --- Get watermark from Delta table (last successful ETL) ---
try:
    watermark_row = spark.sql(f"SELECT MAX(updated_at) AS wm FROM {DELTA_TABLE}").collect()[0]
    watermark = watermark_row["wm"] if watermark_row else None
except Exception:
    watermark = None

if watermark:
    # Incremental: only rows updated/created after watermark
    # Use Spark SQL — timestamps stay in Spark's timezone (UTC), no JDBC mismatch
    pg_df = spark.sql(f"""
        SELECT * FROM {PG_TABLE}
        WHERE COALESCE(updated_at, created_at) > '{watermark.strftime("%Y-%m-%d %H:%M:%S.%f")}'
    """)
    print(f"💧 Watermark: {watermark}")
    print("Incremental load: rows after watermark")
else:
    # Full load (first run or Delta empty)
    pg_df = spark.sql(f"SELECT * FROM {PG_TABLE}")
    print("📦 Full load (no watermark — first run)")

pg_count = pg_df.count()
print(f"Read {pg_count} rows from Lakebase Postgres ✅")
if pg_count > 0:
    display(pg_df.limit(10))

In [0]:
# --- Add etl_date and op_type to source data ---
from pyspark.sql.functions import current_timestamp, lit
pg_df = pg_df.withColumn("etl_date", current_timestamp())
pg_df = pg_df.withColumn("op_type", lit("INSERT"))

# --- Create Delta table if not exists ---
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DELTA_TABLE} (
        video_id       STRING,
        title          STRING,
        description    STRING,
        channel_id     STRING,
        channel_title  STRING,
        published_at   STRING,
        thumbnail_url  STRING,
        video_url      STRING,
        view_count     INT,
        like_count     INT,
        created_at     TIMESTAMP,
        updated_at     TIMESTAMP,
        etl_date       TIMESTAMP,
        op_type        STRING
    )
    USING DELTA
    TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")
print(f"Delta table {DELTA_TABLE} ready ✅")

# Add columns if not exists (for existing tables)
for col_def in ["etl_date TIMESTAMP", "op_type STRING"]:
    col_name = col_def.split()[0]
    try:
        spark.sql(f"ALTER TABLE {DELTA_TABLE} ADD COLUMN {col_def}")
        print(f"Added {col_name} column ✅")
    except Exception:
        print(f"{col_name} column already exists ✅")

# --- MERGE via SQL (compact, op_type tracking) ---
pg_df.createOrReplaceTempView("pg_source")

spark.sql(f"""
    MERGE INTO {DELTA_TABLE} AS t
    USING pg_source AS s ON t.video_id = s.video_id
    WHEN MATCHED THEN UPDATE SET
        title=s.title, description=s.description, channel_id=s.channel_id,
        channel_title=s.channel_title, published_at=s.published_at,
        thumbnail_url=s.thumbnail_url, video_url=s.video_url,
        view_count=s.view_count, like_count=s.like_count,
        updated_at=s.updated_at, etl_date=s.etl_date, op_type='UPDATE'
    WHEN NOT MATCHED THEN INSERT *
""")

merged_count = spark.sql(f"SELECT COUNT(*) FROM {DELTA_TABLE}").collect()[0][0]
print(f"✅ Merged into {DELTA_TABLE}: {merged_count} total rows")

In [0]:
# --- Verify ---
total_delta = spark.sql(f"SELECT COUNT(*) as cnt FROM {DELTA_TABLE}").collect()[0]["cnt"]
with_stats = spark.sql(f"SELECT COUNT(*) as cnt FROM {DELTA_TABLE} WHERE view_count IS NOT NULL").collect()[0]["cnt"]
last_update = spark.sql(f"SELECT MAX(updated_at) as latest FROM {DELTA_TABLE}").collect()[0]["latest"]
last_etl = spark.sql(f"SELECT MAX(etl_date) as etl FROM {DELTA_TABLE}").collect()[0]["etl"]

op_counts = spark.sql(f"SELECT op_type, COUNT(*) as cnt FROM {DELTA_TABLE} GROUP BY op_type").collect()
op_breakdown = ", ".join(f"{r['op_type']}: {r['cnt']}" for r in op_counts)

print(f"📊 Total videos in Delta: {total_delta}")
print(f"📊 Videos with statistics: {with_stats}")
print(f"📊 Last source update: {last_update}")
print(f"📊 Last ETL date: {last_etl}")
print(f"📊 Op type breakdown: {op_breakdown}")
print(f"📊 Rows loaded this run: {pg_count}")
print("\n--- ETL complete ---")